In [1]:
import torch
import torchvision


In [2]:
print("TORCH Version ",torch.__version__)

TORCH Version  2.11.0+cpu


In [3]:
#load the datasets
from torchvision.datasets import MNIST

In [4]:
#download datasets and for train and test use flags train = True/False
data_train = MNIST(root=r"D:\PROJECTS\LEARNINGS\LEARNING_01\MNIST_\GitUnTrack\torch_mnist_dataset",
                    download=True,train=True)

data_test = MNIST(root=r"D:\PROJECTS\LEARNINGS\LEARNING_01\MNIST_\GitUnTrack\torch_mnist_dataset",
                    download=True,train=False)


In [5]:
#datasets N,img,label
print(type(data_train[0][0])) #PIL image
print(data_train[0][0].size) #Image shape
print(data_train[0][1]) #label

<class 'PIL.Image.Image'>
(28, 28)
5


In [6]:
data_train[0][0]

In [7]:
len(data_train),len(data_test)

(60000, 10000)

In [82]:
#Transform data (Normalization,ToTensor) for ANN 
from torchvision.transforms import Lambda,ToTensor,Compose,Normalize #Lamda - define custom transform for all elements,Compose - Combines multiple transforms

transform = Compose([
    ToTensor(),
    # Lambda(lambda image : image/255),
    Normalize((0.1307,), (0.3081,)) # we use standard dataset mnist  mean - 0.1307 and SD - 0.3081 
])

In [83]:
# we can do transform while downloading itself thus save time and space
data_train = MNIST(root=r"D:\PROJECTS\LEARNINGS\LEARNING_01\MNIST_\GitUnTrack\torch_mnist_dataset",
                    download=True,train=True,transform = transform)

data_test = MNIST(root=r"D:\PROJECTS\LEARNINGS\LEARNING_01\MNIST_\GitUnTrack\torch_mnist_dataset",
                    download=True,train=False,transform= transform)

In [84]:
data_train[0][0].shape

torch.Size([1, 28, 28])

In [16]:
type(data_train[0][0]),type(data_train[0][1])

(torch.Tensor, int)

In [32]:
#Model Build 
from torch import nn,optim
import torch.nn.functional as nnF 

In [25]:
#dilation - represents spaced kernels for increasing the convolution area without kernel size params defuault is 1

#1d 
#output = (input+(2*padding)-dilation*(kernel_size-1)-1)/stride + 1  #same can be extended for each h,w for 2D

def find_output_shape(input_shape=(28,28),padding=(0,0),kernel_size=(3,3),stride = (1,1),dilation=(1,1)):
    output = [0,0]
    output[0] = (input_shape[0]+(2*padding[0])-dilation[0]*(kernel_size[0]-1)-1)/stride[0] + 1
    output[1] = (input_shape[1]+(2*padding[1])-dilation[1]*(kernel_size[1]-1)-1)/stride[1] + 1
    return output

In [31]:
find_output_shape(input_shape=(11,11),padding=(0,0),kernel_size=(2,2),stride = (2,2))

[5.5, 5.5]

In [78]:
class CNNModel(nn.Module):
    def __init__(self,batch_size):
        super().__init__()
        
        #conv2d requires (in_channels- input channels count (1-Grayscale,3-RGB),out_channels-filters count,k/(k,k)- kernel_size(int or tuple),1/(1,1) -strides(int or tuple),0/(0,0) padding-int or tuple or str)
        #pool2d requires (k/(k,k)- kernel size int or tuple ,1/(1,1)-strides-int or tuple,0/(0,0) - padding, ceil_mode - determines output shape compute for ceil mode default is True (False-floor output shape compute) )
        #Conv2d (str, optional) – 'zeros', 'reflect', 'replicate' or 'circular'. Default: 'zeros'
        #stride is default  1 for both conv2d and pool2d
        #padding - ph,pw specifies number of padding added in rows horizontally and vertically (ph,pw)

        self.cnn_layer_1 = nn.Conv2d(in_channels = 1,kernel_size=3,out_channels=32,) #input - (batch_size,1,28,28)  output - (32,26,26)
        self.max_pool_layer_1 = nn.MaxPool2d(kernel_size=2,stride=2) #input - (32,26,26) output - (32,13,13)

        self.cnn_layer_2 = nn.Conv2d(in_channels = 32,kernel_size=3,out_channels=16,) #input - (32,13,13) output - (16,11,11)
        self.max_pool_layer_2 = nn.MaxPool2d(kernel_size=2,stride=2) #input - (16,11,11) output - (16,5,5) #5.5 rounded to 5 as defualt 

        self.hidden_layer_1 = nn.Linear(in_features=400, out_features=64) #input after flatten (400) output - 64
        self.output_layer = nn.Linear(in_features=64, out_features=10)

        self.loss = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.parameters()) #parameters are from super class nn.Module (based on current network definition)
        self.batch_size = batch_size

    def forward(self,inputs):
        x = self.cnn_layer_1(inputs)
        x = nnF.relu(x)
        x = self.max_pool_layer_1(x)
        x = self.cnn_layer_2(x)
        x = nnF.relu(x)
        x = self.max_pool_layer_2(x)
        x = torch.flatten(x,1)
        x = self.hidden_layer_1(x)
        x = nnF.relu(x)
        x = self.output_layer(x)
        x = nnF.log_softmax(x, dim=1)
        return x 


    def fit(self,X,Y):
        self.optimizer.zero_grad() #initialize weights grads(not weight values) to zero for fresh compute for each epoch
        y_pred = self.forward(X)
        # loss = self.loss(y_pred,Y)
        loss  = nnF.nll_loss(y_pred, Y)
        loss.backward() #calculation of gradients - backward propagation 
        self.optimizer.step() #update weights
        return loss.item()  #returns only loss values

    def predict(self,X):
        with torch.no_grad():
            return torch.argmax(self.forward(X),axis=1)

    def evaluate(self,test_dataloader):
        correct = 0
        for X,Y in test_dataloader:
            Y_pred = self.predict(X)
            correct += (Y==Y_pred).sum()
        accuracy = correct / (len(test_dataloader)*self.batch_size)
        print(f"Accuracy : {accuracy}")

In [79]:
batch_size = 16
cnn_model = CNNModel(batch_size = batch_size)

In [72]:
# use data  loader to load datasets as batches to nn (provides func as shuffle)
from torch.utils.data import DataLoader 

In [85]:
train_dataloader = DataLoader(data_train,batch_size = batch_size,shuffle=True)
test_dataloader = DataLoader(data_test,batch_size = batch_size,shuffle=True)

In [86]:
next(iter(train_dataloader))[0].shape, len(next(iter(train_dataloader))[1]) #dataloader contains is object class contains the dataset object with Batchsize,data_shape,with labels 

(torch.Size([16, 1, 28, 28]), 16)

In [87]:
from tqdm import tqdm 

In [88]:
epochs = 10
for i in range(epochs):
    total_loss = 0
    for X,Y in tqdm(train_dataloader,desc=f"Fitting NN : Epoch  : {i+1}"):
        loss = cnn_model.fit(X,Y)
        total_loss +=  loss
    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {i+1} Loss :  {avg_loss:.4f}")#loss value should decrease over epoch


Fitting NN : Epoch  : 1: 100%|██████████| 3750/3750 [00:20<00:00, 185.31it/s]


Epoch 1 Loss :  0.1102


Fitting NN : Epoch  : 2: 100%|██████████| 3750/3750 [01:01<00:00, 60.94it/s] 


Epoch 2 Loss :  0.0456


Fitting NN : Epoch  : 3: 100%|██████████| 3750/3750 [01:38<00:00, 38.04it/s]


...

Epoch 8 Loss :  0.0152


Fitting NN : Epoch  : 9: 100%|██████████| 3750/3750 [00:19<00:00, 192.25it/s]


Epoch 9 Loss :  0.0130


Fitting NN : Epoch  : 10: 100%|██████████| 3750/3750 [00:19<00:00, 195.20it/s]

Epoch 10 Loss :  0.0122


In [89]:
cnn_model.evaluate(test_dataloader)

Accuracy : 0.9868999719619751


In [91]:
#saving model weights only
save_path = r"D:\PROJECTS\LEARNINGS\LEARNING_01\MNIST_\model_weights\mnist_cnn_torch_weights.pth"
torch.save(cnn_model.state_dict(),save_path)

In [93]:
#loading weights only needs initializing  nn architecture class
cnn_model_2 = CNNModel(batch_size = batch_size)
cnn_model_2.load_state_dict(torch.load(save_path)) 

<All keys matched successfully>

In [94]:
cnn_model_2.evaluate(test_dataloader)

Accuracy : 0.9868999719619751


In [95]:
#saving model weights + class architecture
save_path_2 =  r"D:\PROJECTS\LEARNINGS\LEARNING_01\MNIST_\model_weights\mnist_cnn_torch_full.pth"
torch.save(cnn_model,save_path_2)

In [96]:
cnn_model_3 = torch.load(save_path_2,weights_only=False)

In [97]:
cnn_model_3.evaluate(test_dataloader)

Accuracy : 0.9868999719619751
